# **Final Candidate Ranking and Substructure Filtering Summary**

Chemistry-informed filtering of the ten ensemble-selected candidates (manuscript Section 2.2.7): elemental filter (Co/Cu), substructure filters (fumarate, succinate), counterion filter, and active-payload mass fraction. Run from the `notebooks/` directory.


In [41]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

df = pd.read_csv('../data/top_10.csv')

# --- SMARTS templates and elemental constraints ---
FUMARATE_SMARTS  = Chem.MolFromSmarts("[CX3](=O)[CX3]=[CX3][CX3](=O)")  # rigid unsaturated bridge (P02)
SUCCINATE_SMARTS = Chem.MolFromSmarts("O=C(O)CCC(=O)O")                 # extended flexible backbone (P04)
# NOTE: this is tetraethylammonium (4 ethyl groups on N+), not triethylammonium
TEA_SMARTS = Chem.MolFromSmarts("CCN(CC)(CC)CC")

NON_TARGET_METALS = {"Co", "Cu"}
# Ba deliberately excluded: not a target plant micronutrient, and phytotoxic at
# elevated concentration on the same grounds as Co/Cu (see ref [1]) — counting it
# as "active payload" would reward toxicity rather than nutritional value.
TARGET_METALS = {"Fe": 55.845, "Zn": 65.38, "Mn": 54.938}
WATER_MW = 18.015

def hydration_offset(mol, stated_formula):
    """
    Compares the stated Formula column against the formula actually encoded
    by the SMILES. If the difference is exactly n*H2O, returns n (waters of
    crystallization). If the gap doesn't resolve to water, returns None
    (flagged for manual review).
    """
    calc_H = sum(a.GetTotalNumHs() for a in mol.GetAtoms())
    calc_O = sum(1 for a in mol.GetAtoms() if a.GetSymbol() == "O")

    import re
    def parse_HO(f):
        h = re.search(r'H(\d*)', f)
        o = re.search(r'O(\d*)', f)
        h_n = int(h.group(1)) if h and h.group(1) else (1 if h else 0)
        o_n = int(o.group(1)) if o and o.group(1) else (1 if o else 0)
        return h_n, o_n

    stated_H, stated_O = parse_HO(stated_formula)
    dH = stated_H - calc_H
    dO = stated_O - calc_O

    if dH == 0 and dO == 0:
        return 0
    if dH >= 0 and dO >= 0 and dH == 2 * dO and dO > 0:
        return dO  # n water molecules (H2O adds +2H +1O each)
    return None  # not explained by hydration — needs manual check

def apply_rdkit_filters(row):
    mol = Chem.MolFromSmiles(row["SMILES"])
    if not mol:
        return False, "Invalid SMILES", 0.0, None

    atoms = [atom.GetSymbol() for atom in mol.GetAtoms()]
    n_water = hydration_offset(mol, row["Formula"])

    # Filter 1: non-target metals (Co, Cu)
    if any(metal in NON_TARGET_METALS for metal in atoms):
        return False, "Filter 1: Non-target metal (Co/Cu)", 0.0, n_water

    # Filter 2: unsaturated rigid dicarboxylate bridges (fumarates)
    if mol.HasSubstructMatch(FUMARATE_SMARTS):
        return False, "Filter 2: Unsaturated rigid dicarboxylate (fumarate)", 0.0, n_water

    # Filter 3: extended flexible chains (succinates, 7-membered chelate ring)
    if mol.HasSubstructMatch(SUCCINATE_SMARTS):
        return False, "Filter 3: Extended carbon chain backbone (succinate)", 0.0, n_water

    # Filter 4: counterion mass ballast in mixed Fe-Mn systems
    if ("Mn" in atoms) and mol.HasSubstructMatch(TEA_SMARTS):
        return False, "Filter 4: Counterion mass ballast (Et4N+ vs NH4+)", 0.0, n_water

    # --- Active metal mass fraction ---
    mw_anhydrous = Descriptors.MolWt(mol)
    mw_used = mw_anhydrous + (n_water or 0) * WATER_MW

    active_mass = sum(TARGET_METALS[m] for m in atoms if m in TARGET_METALS)
    assay_density = active_mass / mw_used if mw_used > 0 else 0.0

    return True, "Passed structural and assay filters", assay_density, n_water

results = df.apply(apply_rdkit_filters, axis=1)
df["Passed_Filters"] = [r[0] for r in results]
df["Filter_Reason"]  = [r[1] for r in results]
df["Assay_Density"]  = [round(r[2], 4) for r in results]
df["H2O_in_formula"] = [r[3] if r[3] is not None else "CHECK MANUALLY" for r in results]
df["Framework_Status"] = df["Passed_Filters"].apply(lambda x: "Selected (Top 5)" if x else "Deferred")

df_sorted = df.sort_values("Assay_Density", ascending=False)
print(df_sorted[["ID", "Formula", "H2O_in_formula", "Assay_Density", "Filter_Reason", "Framework_Status"]].to_string(index=False))

 ID        Formula  H2O_in_formula  Assay_Density                                        Filter_Reason Framework_Status
P15 C3H13FeNO9P3Zn               0         0.2877                  Passed structural and assay filters Selected (Top 5)
P20  C8H12FeMnNO14               2         0.2424                  Passed structural and assay filters Selected (Top 5)
P10   C4H12FeN2O10               2         0.1837                  Passed structural and assay filters Selected (Top 5)
P09   C14H28FeNO10               2         0.1310                  Passed structural and assay filters Selected (Top 5)
P13  C9H22BaFeNO18               6         0.0893                  Passed structural and assay filters Selected (Top 5)
P07 C14H20FeMnNO12               0         0.0000    Filter 4: Counterion mass ballast (Et4N+ vs NH4+)         Deferred
P19 C12H24CuFe2O16               3         0.0000                   Filter 1: Non-target metal (Co/Cu)         Deferred
P17 C2H14CoFeO10P2               4      